# Chapter 8 — Vector Stores & Retrieval

Hands-On LangChain. Code targets **LangChain 1.x**.

Once documents are loaded and split, we need a way to find the chunks relevant to a question. That is
what **embeddings** and **vector stores** give us: text is turned into vectors, similar text lands close
together, and we retrieve by nearness. This chapter covers embeddings, building a vector store, plain
similarity search and where it fails, and then smarter retrieval — **MMR**, **metadata filtering /
self-query**, and **contextual compression**.

Companion notebook for Chapter 8. Requires `OPENAI_API_KEY` and `MODEL`/`EMBEDDING_MODEL` in `.env`.

```bash
pip install langchain-openai langchain-chroma langchain-classic lark -q
```

## 8.0 Setup

In [1]:
import os
import numpy as np
from dotenv import load_dotenv, find_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

load_dotenv(find_dotenv(usecwd=True))
llm_model = os.getenv("MODEL", "gpt-5.4-mini")
embedding_model = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

embedding = OpenAIEmbeddings(model=embedding_model)

## 8.1 What are text embeddings?

An embedding turns a piece of text into a vector of numbers. Text with similar *meaning* gets similar
vectors, so we can measure how related two pieces of text are by how close their vectors are. Let's see
it with three sentences — the first two are both about pets, the third is about a car.

In [2]:
sentence1 = "My dog Rover likes to chase squirrels"
sentence2 = "Fluffy my cat, refuses to eat from a can"
sentence3 = "The chevy bolt accelerates to 60 mph in 6.7 seconds"

embedding1 = embedding.embed_query(sentence1)
embedding2 = embedding.embed_query(sentence2)
embedding3 = embedding.embed_query(sentence3)
print("vector length:", len(embedding1))

vector length: 1536


We compare vectors with **cosine similarity** (higher = more similar). The two pet sentences should score highest.

In [3]:
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("pets   (1 vs 2):", round(cosine(embedding1, embedding2), 4))
print("pet/car(1 vs 3):", round(cosine(embedding1, embedding3), 4))
print("pet/car(2 vs 3):", round(cosine(embedding2, embedding3), 4))

pets   (1 vs 2): 0.2685
pet/car(1 vs 3): 0.1386
pet/car(2 vs 3): -0.0273


## 8.2 Building a vector store

A **vector store** holds these embeddings and lets us look up the nearest ones to a query. We'll use
**Chroma**. For a realistic example we use a small set of course-lecture chunks, each tagged with
metadata (`source` and `page`). Note the first chunk is **duplicated on purpose** — simulating messy
data, which we'll see bite us shortly.

In [4]:
from langchain_core.documents import Document

matlab = ("Those homeworks will be done in either MATLAB or in Octave, which is sort of a free "
          "version of MATLAB. MATLAB makes it very easy to write code using matrices, to write "
          "numerical routines, to move data around and to plot data. Octave is a free alternative "
          "with fewer features that works for the purposes of this class.")

raw_docs = [
    Document(page_content="cs229-qa@cs.stanford.edu. This email goes to an account read by all the "
             "TAs and me, so you can get answers to your questions quickly.",
             metadata={"source": "lecture-01", "page": 1}),
    Document(page_content=matlab, metadata={"source": "lecture-01", "page": 8}),
    Document(page_content=matlab, metadata={"source": "lecture-01", "page": 8}),  # duplicate on purpose
    Document(page_content="I strongly encourage you to form study groups. Find two or three classmates "
             "to study with and discuss the problem sets, which can be reasonably difficult.",
             metadata={"source": "lecture-01", "page": 2}),
    Document(page_content="Supervised learning is the most common form of machine learning: you are "
             "given labelled examples of inputs and outputs and learn to predict the output.",
             metadata={"source": "lecture-02", "page": 3}),
    Document(page_content="Linear regression fits a straight line to data by minimizing the squared "
             "error. It is the simplest supervised learning algorithm we will study.",
             metadata={"source": "lecture-03", "page": 5}),
    Document(page_content="Logistic regression adapts linear regression for classification by squashing "
             "the output through a sigmoid into a probability between 0 and 1.",
             metadata={"source": "lecture-03", "page": 6}),
    Document(page_content="One thing my students and I do is autonomous helicopter flight. It is very "
             "hard to hand-code a controller, so we use learning algorithms instead.",
             metadata={"source": "lecture-01", "page": 9}),
]
print(len(raw_docs), "documents")

8 documents


In [5]:
from langchain_chroma import Chroma

vectordb = Chroma.from_documents(documents=raw_docs, embedding=embedding)
print("stored vectors:", vectordb._collection.count())

stored vectors: 8


Now query by meaning with `similarity_search`. `k=2` returns the two nearest chunks:

In [6]:
question = "What is the email address to reach the course TAs?"
docs = vectordb.similarity_search(question, k=2)
print(docs[0].page_content)

cs229-qa@cs.stanford.edu. This email goes to an account read by all the TAs and me, so you can get answers to your questions quickly.


## 8.3 Where plain similarity search fails

Plain semantic search has two failure modes worth knowing.

**1. Duplicates.** Because we loaded the MATLAB chunk twice, a MATLAB query returns the same text twice
— wasting a retrieval slot on redundant information.

In [7]:
question = "what did they say about matlab?"
docs = vectordb.similarity_search(question, k=3)
print(docs[0].page_content[:90])
print(docs[1].page_content[:90])
print("identical?", docs[0].page_content == docs[1].page_content)

Those homeworks will be done in either MATLAB or in Octave, which is sort of a free versio
Those homeworks will be done in either MATLAB or in Octave, which is sort of a free versio
identical? True


**2. Metadata is ignored.** If we ask specifically about the *third lecture*, semantic search still
mixes in chunks from other lectures, because it matches on meaning alone, not on the structured
`source` field.

In [8]:
question = "what did they say about regression in the third lecture?"
docs = vectordb.similarity_search(question, k=4)
for d in docs:
    print(d.metadata)

{'page': 5, 'source': 'lecture-03'}
{'page': 6, 'source': 'lecture-03'}
{'page': 8, 'source': 'lecture-01'}
{'source': 'lecture-01', 'page': 8}


## 8.4 Maximum Marginal Relevance (MMR)

**MMR** fixes the redundancy problem. Instead of returning only the closest chunks (which may be near-
duplicates), it fetches a larger candidate set (`fetch_k`) and then picks a final `k` that balances
relevance *and* diversity.

A clean illustration: a tiny store of mushroom facts. A plain search for all-white mushrooms returns the
two most similar sentences — and misses the crucial fact that the mushroom is poisonous.

In [9]:
smalldb = Chroma.from_documents(
    documents=[
        Document(page_content="A mushroom with a large fruiting body is the Amanita phalloides. "
                              "Some varieties are all-white."),
        Document(page_content="The Amanita phalloides has a large and imposing epigeous "
                              "(aboveground) fruiting body (basidiocarp)."),
        Document(page_content="A. phalloides, a.k.a Death Cap, is one of the most poisonous "
                              "of all known mushrooms."),
    ],
    embedding=embedding,
)
q = "Tell me about all-white mushrooms with large fruiting bodies"

print("--- similarity_search (k=2) ---")
for d in smalldb.similarity_search(q, k=2):
    print("-", d.page_content)

--- similarity_search (k=2) ---
- A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.
- The Amanita phalloides has a large and imposing epigeous (aboveground) fruiting body (basidiocarp).


In [10]:
print("--- MMR (k=2, fetch_k=3) ---")
for d in smalldb.max_marginal_relevance_search(q, k=2, fetch_k=3):
    print("-", d.page_content)

--- MMR (k=2, fetch_k=3) ---


- A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.
- A. phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.


With MMR, the poisonous-mushroom fact now appears — diversity surfaced information plain similarity missed. The same trick de-duplicates our MATLAB results:

In [11]:
docs_mmr = vectordb.max_marginal_relevance_search("what did they say about matlab?", k=3, fetch_k=6)
for d in docs_mmr[:2]:
    print("-", d.page_content[:90])

- Those homeworks will be done in either MATLAB or in Octave, which is sort of a free versio
- cs229-qa@cs.stanford.edu. This email goes to an account read by all the TAs and me, so you


## 8.5 Metadata filtering and self-query

To honour the "third lecture" part of the question, we filter on metadata. The most direct way is to
pass a `filter` straight to the search — now every result is from `lecture-03`:

In [12]:
docs = vectordb.similarity_search(
    "what did they say about regression?",
    k=3,
    filter={"source": "lecture-03"},
)
for d in docs:
    print(d.metadata, "->", d.page_content[:60])

{'source': 'lecture-03', 'page': 6} -> Logistic regression adapts linear regression for classificat
{'page': 5, 'source': 'lecture-03'} -> Linear regression fits a straight line to data by minimizing


Writing the filter by hand is fine, but we can also let an LLM infer it from a natural-language
question using a **self-query retriever**. We describe the metadata fields, and the LLM splits a question
like *"...in the third lecture"* into a semantic query (*regression*) plus a filter (`source = lecture-03`).
In 1.x this retriever lives in `langchain_classic`.

In [13]:
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo

metadata_field_info = [
    AttributeInfo(name="source",
                  description="The lecture the chunk is from, one of lecture-01, lecture-02, lecture-03",
                  type="string"),
    AttributeInfo(name="page", description="The page number within the lecture", type="integer"),
]

llm = ChatOpenAI(model=llm_model, temperature=0)
retriever = SelfQueryRetriever.from_llm(
    llm, vectordb,
    "Lecture notes from a machine learning course",  # document_contents
    metadata_field_info,
)

results = retriever.invoke("what did they say about regression in the third lecture?")
for d in results:
    print(d.metadata, "->", d.page_content[:60])

{'source': 'lecture-03', 'page': 6} -> Logistic regression adapts linear regression for classificat
{'page': 5, 'source': 'lecture-03'} -> Linear regression fits a straight line to data by minimizing


## 8.6 Contextual compression

A retrieved chunk may be long while only a sentence or two is actually relevant. **Contextual
compression** runs each retrieved chunk through an LLM that extracts just the relevant pieces, so the
final context is tight. It costs extra LLM calls but sharpens the answer.

In [14]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

def pretty_print(docs):
    print(f"\n{'-'*80}\n".join(f"Document {i+1}:\n{d.page_content}" for i, d in enumerate(docs)))

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectordb.as_retriever(),
)

compressed = compression_retriever.invoke("what did they say about matlab?")
pretty_print(compressed)

Document 1:
Those homeworks will be done in either MATLAB or in Octave, which is sort of a free version of MATLAB. MATLAB makes it very easy to write code using matrices, to write numerical routines, to move data around and to plot data.
--------------------------------------------------------------------------------
Document 2:
Those homeworks will be done in either MATLAB or in Octave, which is sort of a free version of MATLAB. MATLAB makes it very easy to write code using matrices, to write numerical routines, to move data around and to plot data.


## 8.7 Combining techniques

These techniques compose. Setting the base retriever's `search_type="mmr"` makes compression operate on a
diverse (de-duplicated) candidate set — sharp *and* non-redundant results.

In [15]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectordb.as_retriever(search_type="mmr"),
)
compressed = compression_retriever.invoke("what did they say about matlab?")
pretty_print(compressed)

Document 1:
Those homeworks will be done in either MATLAB or in Octave, which is sort of a free version of MATLAB. MATLAB makes it very easy to write code using matrices, to write numerical routines, to move data around and to plot data.


With retrieval understood — finding the right chunks, diversifying them, filtering by metadata, and
compressing them — the next chapter feeds these chunks and the question into an LLM to **answer questions
over your documents**.